# 03 — merge y limpieza

---

## 3.1. Limpieza de Headers de los CSV Bronze

### El problema

Los archivos CSV de Kelmarsh provienen de la plataforma Greenbyte y tienen un formato no estándar:

- Las **primeras 9 líneas** son comentarios de metadatos que empiezan con `#`
- La **línea 10** es el header real, pero contiene **caracteres especiales** problemáticos:
  comas dentro de nombres de columna entre comillas, paréntesis, símbolos `°`, `/`, `+`
- Spark no puede inferir correctamente el esquema con este formato

### La solución

Implementamos un parser que:
1. Localiza la línea 10 (el header real)
2. Elimina las comillas y reemplaza las comas internas por guiones
3. Convierte todos los nombres de columna a `snake_case` limpio (solo alfanuméricos y `_`)
4. Genera una lista de rutas + headers limpios para cargar con Spark en el paso siguiente

In [9]:
import os
import glob
import re

base_dir = os.path.dirname(os.getcwd())
bronze_dir = os.path.join(base_dir, "data", "bronze")

target_years = ["2018", "2019", "2020", "2021"]

def clean_header_line(line):
    result = []
    in_quotes = False
    
    for char in line:
        if char == '"':
            in_quotes = not in_quotes
        elif char == ',' and in_quotes:
            result.append('-')
        else:
            result.append(char)
    
    cleaned = ''.join(result)
    if cleaned.startswith('# '):
        cleaned = cleaned[2:]
    elif cleaned.startswith('#'):
        cleaned = cleaned[1:]
    
    columns = cleaned.split(',')
    clean_columns = []
    for col in columns:
        col_clean = re.sub(r'[^a-zA-Z0-9_ ]', '', col)
        col_clean = col_clean.replace(' ', '_')
        col_clean = re.sub(r'_+', '_', col_clean)
        col_clean = col_clean.strip('_')
        col_clean = col_clean.lower()  

        if col_clean == 'date_and_time':
            col_clean = 'timestamp'

        clean_columns.append(col_clean)
    
    return ','.join(clean_columns)

def get_cleaned_files_info():
    """
    Devuelve lista de tuplas: (ruta_archivo, header_limpio)
    """
    all_files = []
    for year in target_years:
        pattern = os.path.join(bronze_dir, f"Kelmarsh_SCADA_{year}_*", f"Turbine_Data_Kelmarsh_1_*.csv")
        all_files.extend(sorted(glob.glob(pattern)))
    
    print(f"📁 Archivos encontrados: {len(all_files)}")
    
    files_info = []
    
    for file_path in all_files:
        filename = os.path.basename(file_path)
        print(f"\n🔄 Analizando: {filename}")
        
        # Leer solo las primeras 10 líneas para encontrar el header
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_number, line in enumerate(f, 1):
                if line_number == 10 and 'Date and time' in line:
                    cleaned_header = clean_header_line(line)
                    files_info.append({
                        'path': file_path,
                        'filename': filename,
                        'header': cleaned_header,
                        'skip_lines': 9  # saltar las 9 primeras (comentarios)
                    })
                    print(f"   ✅ Header limpio: {cleaned_header[:80]}...")
                    break
    
    print(f"\n✅ Total de archivos listos: {len(files_info)}")
    return files_info

# Variable global para el Script 2
files_info = get_cleaned_files_info()

📁 Archivos encontrados: 4

🔄 Analizando: Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv
   ✅ Header limpio: timestamp,wind_speed_ms,wind_speed_standard_deviation_ms,wind_speed_minimum_ms,w...

🔄 Analizando: Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv
   ✅ Header limpio: timestamp,wind_speed_ms,wind_speed_standard_deviation_ms,wind_speed_minimum_ms,w...

🔄 Analizando: Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv
   ✅ Header limpio: timestamp,wind_speed_ms,wind_speed_standard_deviation_ms,wind_speed_minimum_ms,w...

🔄 Analizando: Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv
   ✅ Header limpio: timestamp,wind_speed_ms,wind_speed_standard_deviation_ms,wind_speed_minimum_ms,w...

✅ Total de archivos listos: 4


---

## 3.2. Carga de Telemetría con Spark

Con los headers limpios, cargamos cada archivo anual como un DataFrame de Spark y los unimos en un único DataFrame consolidado.

**Decisión técnica — `unionByName` con `allowMissingColumns=True`:**  
Los archivos de distintos años pueden tener columnas ligeramente distintas (el firmware de la turbina se actualiza y añade o elimina señales). `unionByName` con esta opción rellena con `null` las columnas ausentes en años anteriores, preservando toda la información disponible sin pérdida.

**Resultado esperado:** ~210.000 filas × 303 columnas


In [10]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from IPython.display import display, HTML
display(HTML("<style>div.output_area pre { max-height: none; }</style>"))

# ==============================================================================
# 1. SPARK SESSION
# ==============================================================================
spark = SparkSession.builder \
    .appName("Kelmarsh-EDA-Notebook") \
    .master("local[6]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# ==============================================================================
# 2. VERIFICAR QUE SCRIPT 1 SE EJECUTÓ
# ==============================================================================
try:
    files_info
except NameError:
    raise NameError("❌ Ejecuta primero el Script 1. La variable 'files_info' no existe.")

print(f"📁 Archivos a cargar: {len(files_info)}")
for info in files_info:
    print(f"   - {info['filename']}")

# ==============================================================================
# 3. CARGAR CADA ARCHIVO CON SPARK (LEYENDO COMO TEXTO, FILTRANDO COMENTARIOS)
# ==============================================================================
spark_dfs = []

for info in files_info:
    file_path = info['path']
    header = info['header']
    
    # Leer como RDD de texto para saltar las líneas de comentarios
    rdd = spark.sparkContext.textFile(file_path)
    
    # Filtrar: quitar líneas que empiecen con # (comentarios) y líneas vacías
    # La línea 10 (header) no empieza con # después de la limpieza del Script 1
    filtered_rdd = rdd.zipWithIndex().filter(lambda x: x[1] >= 9).map(lambda x: x[0])
    
    # Convertir RDD a DataFrame usando el header limpio
    # Primero, reconstruir el CSV con el header limpio como primera línea
    header_rdd = spark.sparkContext.parallelize([header])
    full_rdd = header_rdd.union(filtered_rdd)
    
    # Guardar temporalmente y leer con Spark CSV (más robusto para tipos)
    # O usar directamente from_csv si prefieres evitar disco
    
    # Método alternativo más limpio: leer con wholeTextFiles, procesar, y usar spark.read.csv
    # con una versión temporal del archivo
    
    print(f"\n🔹 Cargando: {info['filename']}")
    
    # OPCIÓN MÁS RÁPIDA: usar spark.read.csv con skipLines (Spark 3.3+)
    # o simplemente leer todo y filtrar después
    
    # Para Spark < 3.3, la forma más rápida es:
    raw_df = spark.read.text(file_path)
    
    # Filtrar líneas de comentarios (empiezan con #)
    data_df = raw_df.filter(~F.col("value").startswith("#"))
    
    # La primera fila ahora es el header original (línea 10 del archivo)
    # Necesitamos parsear el CSV manualmente o usar un truco
    
    # MÉTODO MÁS EFICIENTE: escribir un archivo temporal limpio y leerlo
    temp_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "temp_spark")
    os.makedirs(temp_dir, exist_ok=True)
    
    # Leer archivo original, saltar 9 líneas, poner header limpio
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()[10:]  # saltar las 10 primeras
    
    # Insertar header limpio
    lines.insert(0, header + '\n')
    
    # Escribir temporal
    temp_file = os.path.join(temp_dir, f"temp_{info['filename']}")
    with open(temp_file, 'w', encoding='utf-8') as f:
        f.writelines(lines)
    
    # Leer con Spark (ahora es un CSV limpio estándar)
    sdf = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(temp_file)
    
    spark_dfs.append(sdf)
    print(f"   ✅ {sdf.count()} filas x {len(sdf.columns)} columnas")

# ==============================================================================
# 4. UNIR TODOS LOS DATAFRAMES
# ==============================================================================
if len(spark_dfs) == 1:
    turbine_data_df = spark_dfs[0]
else:
    turbine_data_df = spark_dfs[0]
    for sdf in spark_dfs[1:]:
        turbine_data_df = turbine_data_df.unionByName(sdf, allowMissingColumns=True)

# ==============================================================================
# 5. RESULTADOS
# ==============================================================================
print("\n" + "="*60)
print("VISTA PREVIA DE LAS PRIMERAS 5 FILAS:")
turbine_data_df.show(5, truncate=False)

total_records = turbine_data_df.count()

print(f"\n🎯 Total de registros: {total_records}")
print(f"📊 NÚMERO TOTAL DE COLUMNAS: {len(turbine_data_df.columns)}")

📁 Archivos a cargar: 4
   - Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv
   - Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv
   - Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv
   - Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv

🔹 Cargando: Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv


   ✅ 52560 filas x 299 columnas

🔹 Cargando: Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv
   ✅ 52560 filas x 299 columnas

🔹 Cargando: Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv


   ✅ 52704 filas x 299 columnas

🔹 Cargando: Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv


   ✅ 52560 filas x 303 columnas

VISTA PREVIA DE LAS PRIMERAS 5 FILAS:
+-------------------+------------------+--------------------------------+---------------------+---------------------+-----------------+----------------------+-----------------------------------------+------------------------------+------------------------------+----------------------+-----------------------------------------+------------------------------+------------------------------+------------------------------+------------------+------------------+---------------------------------+----------------------+----------------------+-----------------------------------+------------------------+------------------------+-------------------+--------------------+--------------------+-----------------------+-----------------+-------------------------+-----------------+-------------------------+-------------------------------+-------------------------------------+-------------------------------------+-----------------------

## 3.3. filtrado de columnas necesarias

In [11]:
from pyspark.sql import functions as F

# Columnas base para TODOS los modelos — <10% nulos, disponibles siempre
COLS_BASE = [
    "timestamp",
    # Viento
    "wind_speed_ms", "wind_speed_standard_deviation_ms",
    "wind_speed_sensor_1_ms", "wind_speed_sensor_2_ms",
    "wind_direction", "wind_direction_standard_deviation",
    "nacelle_position", "nacelle_position_standard_deviation",
    "vane_position_12",
    # Potencia
    "power_kw", "power_standard_deviation_kw",
    "power_factor_cosphi", "reactive_power_kvar",
    "grid_voltage_v", "grid_frequency_hz",
    # Generador y tren
    "generator_rpm_rpm", "generator_rpm_standard_deviation_rpm",
    "rotor_speed_rpm",
    "drive_train_acceleration_mmss",
    # Temperaturas
    "generator_bearing_front_temperature_c", "generator_bearing_rear_temperature_c",
    "generator_bearing_front_temperature_max_c", "generator_bearing_rear_temperature_max_c",
    "nacelle_temperature_c", "nacelle_temperature_max_c",
    "nacelle_ambient_temperature_c",
    "ambient_temperature_converter_c",
    "front_bearing_temperature_c", "rear_bearing_temperature_c",
    "gear_oil_temperature_c",
    "gear_oil_inlet_temperature_c",
    "stator_temperature_1_c",
    "temp_top_box_c",
    # Hidráulico
    "gear_oil_inlet_pressure_bar", "gear_oil_pump_pressure_bar",
    # Cable y pitch
    "cable_windings_from_calibration_point",
    "blade_angle_pitch_position_a", "blade_angle_pitch_position_b", "blade_angle_pitch_position_c",
    "motor_current_axis_1_a", "motor_current_axis_2_a", "motor_current_axis_3_a",
    "temperature_motor_axis_1_c", "temperature_motor_axis_2_c",
    # Partículas metálicas
    "metal_particle_count",
]


# 1. Filtrar columnas que existen
cols_presentes = [c for c in COLS_BASE if c in turbine_data_df.columns]
turbine_data_df = turbine_data_df.select(*cols_presentes)

print(f"Columnas seleccionadas: {len(turbine_data_df.columns)}")

# 2. Calcular % de nulos + NaN por columna (excluyendo timestamp)
total_filas = turbine_data_df.count()

# Separar timestamp del resto
cols_para_nulos = [c for c in cols_presentes if c != "timestamp"]

nulos_df = turbine_data_df.select([
    (
        F.count(F.when(F.col(c).isNull() | F.isnan(F.col(c)), c)) / total_filas * 100
    ).alias(c)
    for c in cols_para_nulos
])

# Mostrar los 10 con más nulos
nulos_row = nulos_df.collect()[0].asDict()
nulos_ordenados = sorted(nulos_row.items(), key=lambda x: x[1], reverse=True)[:10]

print("Nulos + NaN por columna (top 10):")
for col_name, pct in nulos_ordenados:
    print(f"  {col_name}: {pct:.2f}%")

print("\n" + "="*60)
print("VISTA PREVIA DE LAS PRIMERAS 5 FILAS:")
turbine_data_df.show(5, truncate=False)

total_records = turbine_data_df.count()

print(f"\n🎯 Total de registros: {total_records}")
print(f"📊 NÚMERO TOTAL DE COLUMNAS: {len(turbine_data_df.columns)}")



Columnas seleccionadas: 46


Nulos + NaN por columna (top 10):
  nacelle_temperature_max_c: 8.26%
  generator_bearing_front_temperature_max_c: 7.15%
  generator_bearing_rear_temperature_max_c: 7.15%
  gear_oil_inlet_temperature_c: 3.72%
  vane_position_12: 3.71%
  nacelle_temperature_c: 3.71%
  nacelle_ambient_temperature_c: 3.71%
  front_bearing_temperature_c: 3.71%
  rear_bearing_temperature_c: 3.71%
  gear_oil_temperature_c: 3.71%

VISTA PREVIA DE LAS PRIMERAS 5 FILAS:
+-------------------+------------------+--------------------------------+----------------------+----------------------+------------------+---------------------------------+------------------+-----------------------------------+-------------------+------------------+---------------------------+-------------------+-------------------+------------------+-----------------+------------------+------------------------------------+------------------+-----------------------------+-------------------------------------+------------------------------------+-

De ~3.7 GB CSV → ~50 MB Parquet. Se carga en <5 segundos.

In [12]:
import os


base_dir = os.path.dirname(os.getcwd()) 
silver_dir = os.path.join(base_dir, "data", "silver")

os.makedirs(silver_dir, exist_ok=True)

# Guardar
output_path = os.path.join(silver_dir, "turbine_1_telemetry_clean.parquet")
turbine_data_df.write.parquet(output_path, mode="overwrite")

rel_path = os.path.relpath(output_path, base_dir)
print(f"✅ Guardado en ./{rel_path}")


✅ Guardado en ./data/silver/turbine_1_telemetry_clean.parquet


# failure

Carga y prepara el fault_log

In [13]:
import pandas as pd
import os


base_dir = os.path.dirname(os.getcwd())
fault_log_path = os.path.join(base_dir, "data", "silver", "fault_log.csv")

# Leer CSV
faults = pd.read_csv(fault_log_path, parse_dates=["Timestamp start"])

# Redondear a la baja a 10 minutos (floor)
faults["timestamp"] = faults["Timestamp start"]

# Filtrar solo los que son failure target
targets = faults[faults["is_failure_target"] == True].copy()
targets = targets[["timestamp", "Code", "Message", "Status"]].copy()

print(f"Eventos target totales: {len(targets)}")
print(targets["Code"].value_counts().head(10))


Eventos target totales: 572
Code
6052    132
6200     66
6525     37
6635     37
2125     31
8400     24
5720     22
3000     21
716      17
717      17
Name: count, dtype: int64


In [14]:
# Eventos por timestamp
multi_eventos = targets.groupby("timestamp").agg({
    "Code": lambda x: list(x),
    "Message": "count"
}).rename(columns={"Message": "num_eventos"})

multi_eventos = multi_eventos[multi_eventos["num_eventos"] > 1].sort_values("num_eventos", ascending=False)

print(f"Timestamps con múltiples eventos: {len(multi_eventos)}")
print(f"Total eventos en esos timestamps: {multi_eventos['num_eventos'].sum()}")
print(f"\nEjemplos:")
print(multi_eventos.head())

Timestamps con múltiples eventos: 79
Total eventos en esos timestamps: 293

Ejemplos:
                                                                  Code  \
timestamp                                                                
2021-12-19 10:30:00  [6525, 6635, 6530, 6525, 6635, 6525, 6635, 653...   
2021-12-17 22:30:00  [6525, 6635, 6530, 6525, 6635, 6530, 6525, 663...   
2021-12-06 03:00:00  [6525, 6635, 6530, 6525, 6635, 6525, 6635, 653...   
2021-12-22 02:10:00  [6525, 6635, 6530, 6525, 6635, 6525, 6635, 652...   
2021-12-22 02:40:00         [6525, 6635, 6530, 6525, 6635, 6525, 6635]   

                     num_eventos  
timestamp                         
2021-12-19 10:30:00           18  
2021-12-17 22:30:00           18  
2021-12-06 03:00:00           16  
2021-12-22 02:10:00            9  
2021-12-22 02:40:00            7  


agrupar fallos por familias

In [15]:
import os
import pandas as pd

base_dir = os.path.dirname(os.getcwd())

# FAMILIAS
FAULT_FAMILIES = {
    "yaw_cable":   {"codes": [6052, 6200, 6054, 6120, 6300]},
    "brake_hydro": {"codes": [2125, 5720, 5510, 2000, 1860]},
    "generator":   {"codes": [3000, 2550, 2650, 2655, 2674, 8400, 3125]},
    "pitch_bat":   {"codes": [716, 717, 718, 681, 682, 683, 675, 785, 850]},
}

def get_family(code):
    for family, cfg in FAULT_FAMILIES.items():
        if code in cfg["codes"]:
            return family
    return "other"


# AGRUPAR
targets["family"] = targets["Code"].apply(get_family)

targets_grouped = targets.groupby(["timestamp", "family"]).agg({
    "Code": lambda x: ",".join(map(str, sorted(x.unique()))),
    "Message": lambda x: " | ".join(x.unique()),
    "Status": "first",
}).reset_index()

# Añadir conteo
counts = targets.groupby(["timestamp", "family"]).size().reset_index(name="n_events")
targets_grouped = targets_grouped.merge(counts, on=["timestamp", "family"])

print(f"Eventos: {len(targets)} → {len(targets_grouped)} agrupados")
print(f"\nPor familia:\n{targets_grouped['family'].value_counts()}")

Eventos: 572 → 373 agrupados

Por familia:
family
yaw_cable      187
brake_hydro     58
generator       55
other           43
pitch_bat       30
Name: count, dtype: int64


Eliminar familia "other" antes de guardar, pertenecen a familias de fallos decartadas en el estudio de 01_eda_status_and_events:
| Familia | Códigos principales | Eventos | Justificación para incluir |
|---------|-------------------|---------|---------------------------|
| **Yaw / Cable** | 6052, 6200, 6054 | 217 | Mayor frecuencia del dataset; señal de degradación clara y temprana |
| **Generador / Fans** | 3000, 2550, 2650, 2655, 2674, 8400 | 98 | Bien distribuidos 2018–2021; señal térmica progresiva |
| **Freno / Hidráulico** | 2125, 5720, 5510 | 63 | Señal en presión hidráulica días antes; 4 años de cobertura |
| **Pitch / Baterías** | 716, 717, 718, 681–683 | 71 | Patrón estacional recurrente; ventana de degradación larga (2–4 semanas) |
| **Sensores anem/vane** | 6525, 6635, 6530 | 86 | ⚠️ Descartado para ML — todos los eventos en dic 2021 (un solo incidente) |
| **Torre / Vibración** | 4510, 4540, 59 | 26 | ⚠️ Descartado para ML — resolución 10 min incompatible con física del fallo |

In [16]:
targets_grouped = targets_grouped[targets_grouped["family"] != "other"].copy()

print(f"Eventos: {len(targets)} → {len(targets_grouped)} agrupados")
print(f"\nPor familia:\n{targets_grouped['family'].value_counts()}")


Eventos: 572 → 330 agrupados

Por familia:
family
yaw_cable      187
brake_hydro     58
generator       55
pitch_bat       30
Name: count, dtype: int64


comprobación para verificar que los timestamps de los fallos agrupados coinciden con los timestamps de la telemetría SCADA.

In [17]:
from pyspark.sql import functions as F

# En PySpark
telem_times = set(
    turbine_data_df.select(
        F.date_trunc("hour", F.col("timestamp")) + 
        F.floor(F.minute(F.col("timestamp")) / 10) * F.expr("INTERVAL 10 MINUTES")
    )
    .distinct()
    .rdd.flatMap(lambda x: x)
    .collect()
)

fault_times = set(targets_grouped["timestamp"].unique())
overlap = fault_times.intersection(telem_times)

print(f"Eventos target que tienen telemetría: {len(overlap)}/{len(fault_times)}")

Eventos target que tienen telemetría: 316/316


de los 330 fallos agrupados, 316 timestamp encajan con los datos de los sensores. y aun asi son el 100%. se esperan que los 14 fallos de diferencia compartan mismo timestamp con otra familia. a continuacion lo verificamos.

In [18]:
# Ver timestamps duplicados en targets_grouped
duplicados = targets_grouped.groupby("timestamp").size()
duplicados = duplicados[duplicados > 1]

print(f"Timestamps con múltiples familias: {len(duplicados)}")
print(f"Total de duplicados (filas extra): {duplicados.sum() - len(duplicados)}")

print(f"\nEjemplos de timestamps con múltiples familias:")
for ts in duplicados.head(5).index:
    familias = targets_grouped[targets_grouped["timestamp"] == ts]["family"].tolist()
    print(f"  {ts}: {familias}")

Timestamps con múltiples familias: 14
Total de duplicados (filas extra): 14

Ejemplos de timestamps con múltiples familias:
  2018-05-21 16:30:00: ['brake_hydro', 'pitch_bat']
  2018-11-02 16:30:00: ['brake_hydro', 'generator']
  2018-11-02 16:40:00: ['brake_hydro', 'generator']
  2019-11-05 15:50:00: ['brake_hydro', 'generator']
  2019-11-05 16:00:00: ['brake_hydro', 'generator']


Resultado: 14 timestamps tienen múltiples familias de fallo simultáneas.
Ejemplos:

* 2018-05-21 16:30:00: brake_hydro + pitch_bat

* 2018-11-02 16:30:00: brake_hydro + generator

¿Qué significa?

Fallos en cascada: cuando un subsistema falla gravemente, otros fallan casi al mismo tiempo.
No es un problema: cada familia tiene su propio modelo. El mismo timestamp puede ser positivo para múltiples modelos sin conflicto.
No se pierde información: las 15 filas "extra" están en el dataset; simplemente comparten timestamp con otra familia.

Implicación para el modelo:

Modelo yaw_cube: timestamp 2019-03-03 21:40:00 → positivo (hay fallo yaw)
Modelo brake_hydro: mismo timestamp → también positivo (hay fallo freno)
Cada modelo se entrena de forma independiente.
Conclusión: Esto es normal en turbinas eólicas. Continuamos con el etiquetado por familia.

In [25]:
import pandas as pd

# Ajustar el ancho máximo de la pantalla para que no parta la tabla
pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)

#guardar
output_path = os.path.join(base_dir, "data", "silver", "fault_targets_grouped.parquet")
targets_grouped.to_parquet(output_path, index=False)

rel_path = os.path.relpath(output_path, base_dir)
print(f"✅ Guardado en ./{rel_path}")
print(f"Eventos: {len(targets)} → {len(targets_grouped)} agrupados")
print(f"\nPor familia:\n{targets_grouped['family'].value_counts()}")

print(f"\nVISTA PREVIA:")
print(targets_grouped)


✅ Guardado en ./data/silver/fault_targets_grouped.parquet
Eventos: 572 → 330 agrupados

Por familia:
family
yaw_cable      187
brake_hydro     58
generator       55
pitch_bat       30
Name: count, dtype: int64

VISTA PREVIA:
              timestamp       family       Code                            Message   Status  n_events
1   2018-01-11 15:00:00  brake_hydro       2125               Timeout brake closed  Warning         1
3   2018-01-18 06:00:00    generator       8400                  Comm. failure FPM  Warning         1
4   2018-01-18 06:10:00    generator       8400                  Comm. failure FPM  Warning         1
5   2018-01-18 06:20:00    generator       8400                  Comm. failure FPM  Warning         1
6   2018-01-18 07:00:00    generator       8400                  Comm. failure FPM  Warning         1
..                  ...          ...        ...                                ...      ...       ...
359 2021-11-27 13:00:00    yaw_cable       6052             H